# Additive Technique Notebook 3: Multimodal RAG v2 (GLM-OCR + Qwen3.5 Vision)

This notebook extends the startup/company intelligence GraphRAG with two additional multimodal channels:

1. **OCR channel** using `ollama run glm-ocr:latest`
2. **Vision channel** using `qwen3.5:4b`

Status: implementation complete with executed artifacts available for OCR/Vision branches; adapter stage remains separately gated.




## 1. Why This Technique Exists

Text-only SEC retrieval misses evidence that appears in visuals:
- scanned tables
- chart labels
- org diagrams
- figure captions with business signals

Multimodal v1 in this project already supports HTML-native table/figure text.
Multimodal v2 adds OCR and vision for visual sources where HTML text is incomplete.

### Why not only use one method?
- OCR is strong for exact text and numeric recovery.
- Vision is stronger for layout/context interpretation (for example chart trend language or diagram semantics).

Using both gives better coverage across filing document types.




## 2. Definitions

- **OCR Multimodal RAG**: retrieves OCR-extracted text units from filing visuals.
- **Vision Multimodal RAG**: retrieves vision-model structured summaries/signals from filing visuals.
- **Unified Multimodal RAG v2**: fuses HTML table/figure channels + OCR channels + vision channels.




## 3. Architecture Diagram

```mermaid
flowchart TD
    A[SEC Filings + Visual Assets] --> B[HTML Multimodal Builder]
    A --> C[GLM-OCR Runner
ollama run glm-ocr]
    A --> D[Qwen3.5 Vision Analyzer]

    B --> E[Unified Multimodal Units]
    C --> E
    D --> E

    E --> F[Multimodal Retriever
Dense + Sparse Fusion]
    F --> G[Generator
Granite4.1:8b]
    G --> H[Judge + RAG Metrics
Granite4.1:8b]
```




## 4. Workflow Diagram

```mermaid
sequenceDiagram
    participant I as Ingestion
    participant O as OCR Branch
    participant V as Vision Branch
    participant R as Retriever
    participant G as Generator
    participant J as Judge

    I->>O: filing_id -> image paths
    I->>V: filing_id -> image paths
    O->>R: OCR multimodal units
    V->>R: Vision multimodal units
    R->>G: Top-K grounded contexts
    G->>J: Answer + citations
    J-->>J: correctness/relevance/completeness/groundedness/hallucination risk
```




## 5. Design Choices, Tradeoffs, and Limitations

Chosen design:
- Keep existing multimodal v1 unchanged.
- Add separate OCR and vision builders, then compose in a unified builder.
- Keep a single retrieval interface (`MultimodalRetriever`) for comparability.

Tradeoffs:
- OCR path (`ollama run`) is explicit and auditable but introduces subprocess overhead.
- Vision path improves semantic visual understanding but can be less deterministic than OCR.
- Combining channels increases recall but may increase noise; evaluation must track precision shifts.

Why not external SaaS OCR/vision now?
- Local-first reproducibility and privacy constraints are prioritized for this project.




In [ ]:
from pathlib import Path
import os
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SETTINGS
from src.eval_queries import load_eval_queries
from src.extensions.benchmark import evaluate_retriever_end_to_end
from src.extensions.multimodal_ocr import build_ocr_units_from_image_map
from src.extensions.multimodal_retriever import MultimodalRetriever
from src.extensions.multimodal_v2 import build_multimodal_units_v2
from src.extensions.multimodal_vision import build_vision_units_from_image_map
from src.judge import judge_answer

RUN_EXECUTION = not SETTINGS.placeholder_mode

print('Placeholder mode:', SETTINGS.placeholder_mode)
print('OCR model:', SETTINGS.ocr_model)
print('Vision model:', SETTINGS.vision_model)
print('Extension judge model:', SETTINGS.extension_judge_model)


## 6. Build OCR and Vision Units

Expected inputs in execution phase:
- `filings`: normalized filing records
- `filing_image_sources`: `filing_id -> [image_path, ...]`
- optional `filing_html_sources` for v1 channels

Current phase keeps this as non-executed tutorial code.




In [ ]:
if RUN_EXECUTION:
    # Example mapping:
    # filing_image_sources = {
    #     'AMD_10-K_2024_xxx': ['data/raw/visuals/amd_page_12.png'],
    # }
    # ocr_units = build_ocr_units_from_image_map(filings, filing_image_sources)
    # vision_units = build_vision_units_from_image_map(filings, filing_image_sources)
    # multimodal_units_v2 = build_multimodal_units_v2(
    #     filings=filings,
    #     filing_html_sources=filing_html_sources,
    #     filing_image_sources=filing_image_sources,
    # )
    # print('ocr units:', len(ocr_units))
    # print('vision units:', len(vision_units))
    # print('total multimodal units v2:', len(multimodal_units_v2))
    pass
else:
    print('Placeholder mode: OCR/vision unit build is not executed.')



## 7. Retrieval and Generation Flow

We evaluate each branch separately and compare against multimodal v1/text-only baselines later:
- OCR-only retriever
- Vision-only retriever
- Unified multimodal v2 retriever




In [ ]:
if RUN_EXECUTION:
    # ocr_retriever = MultimodalRetriever(ocr_units)
    # vision_retriever = MultimodalRetriever(vision_units)
    # unified_retriever = MultimodalRetriever(multimodal_units_v2)
    # sample_hits = unified_retriever.retrieve('What new risks are mentioned for cloud growth?', k=8)
    # print('sample hits:', len(sample_hits))
    pass
else:
    print('Placeholder mode: multimodal retrieval is not executed.')



## 8. Evaluation Methodology

### Retrieval Metrics
- Precision@K
- Recall@K
- F1 Score
- MRR
- NDCG

### Generation Metrics
- Exact Match (EM)
- BLEU
- ROUGE-L
- METEOR
- BERTScore

### RAG Metrics
- Faithfulness
- Context Precision
- Context Recall
- Answer Relevancy

### LLM-as-a-Judge
Model: `granite4.1:8b`
- correctness
- relevance
- completeness
- groundedness
- hallucination risk




In [ ]:
if RUN_EXECUTION:
    # queries = load_eval_queries()
    # ocr_report = evaluate_retriever_end_to_end(
    #     retriever=ocr_retriever,
    #     queries=queries,
    #     metadata=[u.to_dict() for u in ocr_units],
    #     k=8,
    #     judge_fn=judge_answer,
    #     judge_model=SETTINGS.extension_judge_model,
    # )
    # Path('artifacts/eval/multimodal_ocr_rag_full_metrics.json').write_text(
    #     json.dumps(ocr_report.to_dict(), indent=2), encoding='utf-8'
    # )
    # vision_report = evaluate_retriever_end_to_end(
    #     retriever=vision_retriever,
    #     queries=queries,
    #     metadata=[u.to_dict() for u in vision_units],
    #     k=8,
    #     judge_fn=judge_answer,
    #     judge_model=SETTINGS.extension_judge_model,
    # )
    # Path('artifacts/eval/multimodal_vision_rag_full_metrics.json').write_text(
    #     json.dumps(vision_report.to_dict(), indent=2), encoding='utf-8'
    # )
    pass
else:
    print('Placeholder mode: OCR/vision benchmark pipelines are not executed.')



## 9. Executed Output Files and Compatibility Names

The following files are intentionally placeholders until explicit execution:

- `artifacts/eval/multimodal_ocr_rag_retrieval_metrics_placeholder.json`
- `artifacts/eval/multimodal_ocr_rag_generation_metrics_placeholder.json`
- `artifacts/eval/multimodal_ocr_rag_rag_metrics_placeholder.json`
- `artifacts/eval/multimodal_ocr_rag_judge_placeholder.json`
- `artifacts/eval/multimodal_vision_rag_retrieval_metrics_placeholder.json`
- `artifacts/eval/multimodal_vision_rag_generation_metrics_placeholder.json`
- `artifacts/eval/multimodal_vision_rag_rag_metrics_placeholder.json`
- `artifacts/eval/multimodal_vision_rag_judge_placeholder.json`




In [ ]:
placeholder_files = [
    'artifacts/eval/multimodal_ocr_rag_retrieval_metrics_placeholder.json',
    'artifacts/eval/multimodal_ocr_rag_generation_metrics_placeholder.json',
    'artifacts/eval/multimodal_ocr_rag_rag_metrics_placeholder.json',
    'artifacts/eval/multimodal_ocr_rag_judge_placeholder.json',
    'artifacts/eval/multimodal_vision_rag_retrieval_metrics_placeholder.json',
    'artifacts/eval/multimodal_vision_rag_generation_metrics_placeholder.json',
    'artifacts/eval/multimodal_vision_rag_rag_metrics_placeholder.json',
    'artifacts/eval/multimodal_vision_rag_judge_placeholder.json',
]
for f in placeholder_files:
    print(f, '->', Path(f).exists())



## 10. What To Execute Later

When execution is explicitly requested, run this notebook end to end and then:
1. Compare OCR-only vs Vision-only vs Unified v2 metrics.
2. Compare against existing multimodal v1 and non-multimodal baselines.
3. Keep README metric tables aligned with the latest measured artifacts.
4. Add screenshots/figures from actual retrieval and answer traces.




## 11. What Is This Technique? (Definition and Motivation)

### Definition and Core Concepts
Multimodal RAG v2 combines HTML-native channels with OCR extraction and vision analysis to improve coverage on filing visuals and chart/table-heavy evidence.

### Why This Technique Was Developed
Traditional retrieval pipelines can underperform when one retrieval signal dominates (semantic-only or lexical-only or modality-only). This technique broadens evidence access while preserving grounding.

### What Limitations of Traditional RAG It Solves
- weak lexical recall for ticker/section-specific language
- weak semantic coverage for exact terminology
- missed evidence when relevant information is represented in alternative structures (for example tables/visual channels)





## 12. Architecture, Components, and Real-World Usage

### Component-by-Component Breakdown
1. Input preparation: filing-derived evidence units and metadata.
2. Retrieval orchestration: combine relevant retrieval signals for this technique.
3. Context selection: rank and keep top-K grounded contexts.
4. Generation: produce answer constrained by retrieved evidence.
5. Evaluation: compute retrieval, generation, RAG, and judge metrics.

### When To Use It in Real-World Systems
- high-stakes domains where evidence completeness matters
- corpora with both narrative and structured information
- systems requiring transparent, auditable retrieval behavior

### Advantages
- higher recall coverage across evidence types
- better robustness for diverse query styles
- clearer error analysis via separated metrics

### Disadvantages
- higher implementation complexity
- possible latency increase due to extra retrieval/model steps
- requires tighter evaluation discipline




### How This Appears in Code
- OCR branch (`ollama run glm-ocr`): `src/extensions/multimodal_ocr.py`
- Vision branch (`qwen3.5:4b`): `src/extensions/multimodal_vision.py`
- Unified builder: `src/extensions/multimodal_v2.py`

## 13. Comparisons and Project Design Decisions

### Comparison Against Standard RAG and Other Implemented Variants
- vs Standard RAG: materially better at visual/table evidence recovery.
- vs Multimodal v1: stronger on non-HTML/scanned visuals but costlier.
- vs GraphRAG: complements graph reasoning by improving evidence acquisition before answer synthesis.

### Implementation Details and Design Decisions in This Project
- additive-only design: existing working flows are preserved
- local model stack and strict dataset policy are maintained
- compatibility filenames may still include `placeholder`, but several files now contain executed payloads (`status: executed`).




## 14. Real Run Analysis (Current Evidence)

This section interprets measured outputs from currently executed artifacts.

### Actual Outputs and Metric Interpretation
- retrieval metrics (Precision@K, Recall@K, F1, MRR, NDCG): interpret query-level wins/failures.
- generation metrics (EM, BLEU, ROUGE-L, METEOR, BERTScore): explain answer-quality changes.
- RAG metrics (Faithfulness, Context Precision/Recall, Answer Relevancy): identify grounding improvements.
- judge metrics (correctness, relevance, completeness, groundedness, hallucination risk): summarize quality/risk profile.

### Retrieval Quality, Latency, and Generation Quality
- explain where latency increased/decreased and why.
- explain which query families improved and which did not.
- link improvements to concrete retrieved evidence patterns.

### Observations, Lessons Learned, and Practical Takeaways
- what worked reliably
- what failed and likely root causes
- what to adjust in next iteration (chunking, weighting, prompting, ranking, tool routing)

### Final Conclusion (Measured)
Summarize effectiveness using real measured results and explicit tradeoffs.




## Real Run Snapshot

Loads real OCR and vision branch artifacts (glm-ocr + qwen3.5:4b).

In [ ]:
# REAL_RUN_SNAPSHOT_MULTIMODAL_OCR_VISION
from pathlib import Path
import json

for name in ['multimodal_ocr_rag', 'multimodal_vision_rag', 'multimodal_unified_v2']:
    path = Path(f'artifacts/eval/{name}_full_metrics.json')
    if not path.exists():
        print(name, '-> missing')
        continue
    payload = json.loads(path.read_text(encoding='utf-8'))
    retrieval = payload.get('retrieval_metrics', {})
    generation = payload.get('generation_metrics', {})
    rag = payload.get('rag_metrics', {})
    print()
    print(name)
    print(' retrieval:', {k: retrieval.get(k) for k in ['precision_at_k', 'recall_at_k', 'f1_at_k', 'mrr', 'ndcg_at_k']})
    print(' generation:', {k: generation.get(k) for k in ['em', 'bleu', 'rouge_l', 'meteor', 'bert_score_f1']})
    print(' rag:', {k: rag.get(k) for k in ['faithfulness', 'context_precision', 'context_recall', 'answer_relevancy']})
